# 10주차 ② RNN 과 LSTM — 실습 4

**목표**: RNN 의 순환 구조와 **장기 의존성 문제**를 이해하고,
LSTM 감성 분류 모델을 학습시켜 3교시 비교의 기준선을 만든다.

```
   아이디어 : 앞 단어를 읽은 "요약"을 다음 단어로 넘긴다

        h₀ ──→ [RNN] ──→ h₁ ──→ [RNN] ──→ h₂ ──→ [RNN] ──→ h₃
                 ↑                ↑                ↑
                이               영화              별로

        hₜ = tanh(Wₕ·hₜ₋₁ + Wₓ·xₜ + b)
```

> **핵심 메시지 ★**: **가중치가 하나뿐입니다.** 40 시점을 처리해도 `Wₕ`, `Wₓ` 는 한 벌입니다.
> 7주차 CNN 에서 *"같은 커널을 모든 위치에 재사용"* 한 것과 **같은 발상**입니다.

### 장기 의존성 문제 ★★

```
   h₁ → h₂ → h₃ → ... → h₄₀        매 시점 Wₕ 를 곱한다. 40번 곱하면?

          0.9⁴⁰ ≈ 0.015        ← 앞 정보가 거의 사라진다 (기울기 소실)
          1.1⁴⁰ ≈ 45           ← 반대로 폭주하기도 한다 (기울기 폭발)
```

> **핵심 메시지 ★★ (기말 출제 1순위)**: 시점이 길어지면 **앞쪽 정보가 뒤까지 전달되지 않습니다.**
> 원인은 **같은 가중치를 반복해서 곱하기 때문**이고,
> 이건 **5주차의 기울기 소실과 같은 뿌리**입니다.
> (5주차에는 층을 5개 쌓아서, 오늘은 시점을 40번 반복해서.)

```
   "이 영화는 배우도 좋고 영상미도 훌륭하고 음악도 인상적인데 결말이 최악이다"

        판정에 결정적인 단어 : "최악"  (문장 끝)
        그런데 앞의 칭찬 단어들이 h 를 이미 긍정 쪽으로 밀어 놨다
```

In [ ]:
# 셀 1 — RNN 의 출력은 두 개다  ★ 3교시의 복선
import torch, torch.nn as nn

e = torch.randn(2, 5, 128)                            # (batch, seq_len, embed)
rnn = nn.RNN(input_size=128, hidden_size=128, batch_first=True)
out, h = rnn(e)
print("RNN  | out :", out.shape, " ← 모든 시점의 은닉 상태")
print("     | h   :", h.shape,   " ← 마지막 시점만")

lstm = nn.LSTM(128, 128, batch_first=True)
out2, (h2, c2) = lstm(e)                              # ★ LSTM 은 (h, c) 튜플
print("\nLSTM | out :", out2.shape, "| h :", h2.shape, "| c :", c2.shape)

print("\nout 의 마지막 시점과 h 가 같은가? :", torch.allclose(out2[:, -1, :], h2[0]))

> **핵심 메시지 ★**: 출력이 **두 개**입니다.
> - `out` : **모든 시점**의 은닉 상태
> - `h` : **마지막 시점**만
>
> 오늘 2교시는 **`h` 만** 씁니다. **3교시에 `out` 을 쓰게 됩니다** — 그게 어텐션입니다.

> **함정 ★**: RNN 은 `out, h`, LSTM 은 **`out, (h, c)`** 입니다.
> 이걸 헷갈리면 `too many values to unpack` 오류가 납니다.
> 그리고 `batch_first=True` 를 빠뜨리면 shape 이 `(seq_len, batch, hidden)` 이 됩니다.

In [ ]:
# 셀 2 — 반복해서 곱하면 어떻게 되나 (장기 의존성을 숫자로)
for scale in [0.9, 1.0, 1.1]:
    vals = [scale**t for t in [1, 5, 10, 20, 40]]
    print(f"  {scale} 를 t번 곱하면 : " + " ".join(f"{v:8.3f}" for v in vals))
print("\n  t =            1        5       10       20       40")
print("\n→ 0.9 쪽은 사라지고(소실), 1.1 쪽은 폭주합니다(폭발).")
print("  둘 사이의 아주 좁은 길만 안전합니다 — 그래서 RNN 은 길면 어렵습니다.")

### LSTM 게이트

```
   RNN  : 매 시점 h 를 통째로 갈아엎는다      →  옛 정보가 덮인다
   LSTM : 별도의 "장기 기억" c 를 두고,
          무엇을 지우고 / 넣고 / 꺼낼지를 게이트가 결정한다   ★ 그 결정도 학습된다
```

| 게이트 | 하는 일 |
|---|---|
| **망각 게이트 (forget)** | 장기 기억 `c` 에서 **무엇을 지울지** |
| **입력 게이트 (input)** | 새 정보 중 **무엇을 넣을지** |
| **출력 게이트 (output)** | `c` 에서 **무엇을 꺼내 `h` 로 쓸지** |

```
        cₜ₋₁ ──(× 망각)──(+ 입력)──→ cₜ        ★ c 는 곱셈이 아니라 덧셈으로 이어진다
                                       │
                                    (× 출력)
                                       ↓
                                      hₜ
```

> **핵심 메시지 ★**: 핵심은 **`c` 가 덧셈으로 흐른다**는 것입니다.
> RNN 은 매번 곱해서 정보가 지수적으로 줄었는데, LSTM 은 **더하면서 지나가므로** 멀리 갑니다.
> **7주차 ResNet 의 `+x` 와 같은 발상**입니다 — 지름길을 놓아 신호를 살린다.

> **그래도 완전하지는 않습니다.** LSTM 은 장기 의존성을 **완화**했지만 **해결하지는 못했습니다.**
> 근본 문제는 그대로입니다 — *"문장 전체를 벡터 하나에 담는다"*. **3교시가 그 가정을 버립니다.**

### GRU 는 무엇이 다른가

| | LSTM | GRU |
|---|---|---|
| 게이트 | 3개 (망각·입력·출력) | **2개** (업데이트·리셋) |
| 상태 | `h` 와 `c` 두 개 | **`h` 하나** |
| 파라미터 | 많다 | **약 3/4** |
| 성능 | 대체로 비슷 | 대체로 비슷 |

## 실습 4 — LSTM 감성 분류 학습

In [ ]:
# 셀 3 — 1교시에서 이어서 (커널을 재시작했다면 이 셀부터)
import time
from torch.utils.data import TensorDataset, DataLoader
from preprocess_text import load_nsmc, load_vocab, encode

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치 :", device)

train_texts, train_labels, val_texts, val_labels = load_nsmc("data/nsmc_subset")
vocab = load_vocab("models/vocab.json")               # ★ 1교시에서 저장한 것
VOCAB_SIZE, EMBED, HIDDEN, MAX_LEN = len(vocab), 128, 128, 40

X  = torch.tensor([encode(t, vocab, MAX_LEN) for t in train_texts])
y  = torch.tensor(train_labels)
Xv = torch.tensor([encode(t, vocab, MAX_LEN) for t in val_texts])
yv = torch.tensor(val_labels)
print("훈련 :", X.shape, "| 검증 :", Xv.shape)

In [ ]:
# 셀 4 — 모델 정의
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB_SIZE, EMBED, padding_idx=0)
        self.lstm = nn.LSTM(EMBED, HIDDEN, batch_first=True)
        self.drop = nn.Dropout(0.3)                       # 6주차
        self.fc   = nn.Linear(HIDDEN, 2)                  # 부정/긍정

    def forward(self, x):
        e = self.emb(x)                    # (B, L, EMBED)
        out, (h, c) = self.lstm(e)         # out: (B, L, HIDDEN) / h: (1, B, HIDDEN)
        last = h[-1]                       # ★ 마지막 시점만 쓴다  (B, HIDDEN)
        return self.fc(self.drop(last))

torch.manual_seed(0)
model = LSTMClassifier().to(device)
print(model)
print("파라미터 수 :", sum(p.numel() for p in model.parameters()))
print("  그중 임베딩 :", model.emb.weight.numel(), " ← 대부분이 여기다")

dummy = torch.zeros(4, MAX_LEN, dtype=torch.long).to(device)
print("출력 shape :", model(dummy).shape)          # (4, 2)

> **관찰 포인트 ★★**: `forward` 의 **`last = h[-1]`** 에 밑줄을 치세요.
> **`out`(모든 시점)을 계산해 놓고 버리고, 마지막 하나만 씁니다.**
> 3교시에 **바로 이 줄을 바꿉니다.**

> 파라미터의 대부분이 **임베딩**입니다 (20,000 × 128 = 256만).
> LSTM 자체는 그보다 훨씬 작습니다.

In [ ]:
# 셀 5 — 학습 (루프는 6주차 그대로)
train_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=128)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 5

def evaluate(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"epoch {epoch+1}/{EPOCHS} | 검증 정확도 {evaluate(model)*100:5.2f}%")

lstm_acc = evaluate(model)
print(f"\nLSTM 최종 : {lstm_acc*100:.2f}%  ({time.time()-t0:.1f}초)")

In [ ]:
# 셀 6 — 저장 (3교시에서 비교에 쓴다)
import os, json
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
torch.save(model.state_dict(), "models/lstm_only.pt")
json.dump({"lstm_acc": lstm_acc}, open("results/lstm.json", "w"), ensure_ascii=False)
print("저장 완료 : models/lstm_only.pt, results/lstm.json")

> **핵심 메시지 ★**: **학습 루프가 6주차와 똑같습니다.**
> 5주차 MLP, 7주차 CNN, 9주차 ViT, 오늘 LSTM — **바뀐 건 모델 구조뿐**입니다.
> 4주차에 손으로 쓴 그 다섯 줄이 학기 내내 그대로입니다.

In [ ]:
# 셀 7 — 직접 문장을 넣어 본다
def predict(text):
    idx = torch.tensor([encode(text, vocab, MAX_LEN)]).to(device)
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(idx), dim=1)[0]
    return f"부정 {p[0]*100:5.1f}% / 긍정 {p[1]*100:5.1f}%"

for s in ["정말 재미있었다",
          "시간이 아까웠다",
          "배우도 좋고 영상도 훌륭한데 결말이 최악이다"]:      # ★ 세 번째를 주목
    print(f"{s:40s} → {predict(s)}")

> **관찰 포인트 ★★**: **세 번째 문장**을 보세요. 사람은 "최악" 때문에 부정으로 읽습니다.
> **모델은 어떻게 답했나요?** 헷갈렸다면 **장기 의존성 문제**가 눈앞에서 벌어진 것입니다.
> 3교시 어텐션이 이걸 개선하는지 확인합니다.

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `too many values to unpack` | `out, h = lstm(e)` 로 썼다. LSTM 은 `out, (h, c)` ★ |
> | shape 오류 | `batch_first=True` 를 빠뜨렸다 |
> | 정확도가 50%에서 안 오른다 | 라벨이 섞였거나 lr 문제. `1e-3` 확인 |
> | `IndexError` in Embedding | 인덱스가 `vocab_size` 를 넘었다. 어휘사전 확인 |

In [ ]:
# 셀 8 (예비 · 3분이 남으면) — GRU 로 바꿔 1 epoch 만 비교
class GRUClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED, padding_idx=0)
        self.gru = nn.GRU(EMBED, HIDDEN, batch_first=True)   # ★ c 가 없다
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(HIDDEN, 2)

    def forward(self, x):
        out, h = self.gru(self.emb(x))                       # ★ RNN 과 같은 형태
        return self.fc(self.drop(h[-1]))

torch.manual_seed(0)
g = GRUClassifier().to(device)
opt = torch.optim.Adam(g.parameters(), lr=1e-3)
g.train()
for xb, yb in train_loader:
    xb, yb = xb.to(device), yb.to(device)
    loss = loss_fn(g(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"LSTM 파라미터 : {sum(p.numel() for p in model.lstm.parameters()):,}")
print(f"GRU  파라미터 : {sum(p.numel() for p in g.gru.parameters()):,}  ← 약 3/4")
print(f"GRU 1 epoch 검증 정확도 : {evaluate(g)*100:.2f}%")

> **포인트**: *"게이트를 줄여도 크게 안 떨어진다"* 가 GRU 의 존재 이유입니다.
> 여기까지만 알면 충분합니다.

---

### 이 노트북 체크리스트

- [ ] RNN 이 같은 가중치를 재사용한다는 것을 안다
- [ ] **장기 의존성 문제의 원인**을 말할 수 있다 ★★
- [ ] 그것이 5주차 기울기 소실과 같은 뿌리임을 안다
- [ ] LSTM 게이트 3개가 각각 무엇을 하는지 안다
- [ ] `c` 가 덧셈으로 흐르는 것이 ResNet 과 닮았다는 것을 안다
- [ ] GRU 와 LSTM 의 차이를 한 줄로 말할 수 있다
- [ ] LSTM 감성 분류를 학습시키고 정확도를 기록했다
- [ ] **`last = h[-1]` 이 무엇을 버리고 있는지** 안다 ★